# Initial Steps: load packages, files and functions

In [1]:
import json, re, time, itertools
from copy import deepcopy
from pathlib import Path
from datetime import datetime
import os
import pandas as pd
import requests

In [2]:
cwd=os.getcwd()
cwd_Raw_Data_outputs=os.path.join(cwd,'RawData')#heres where we store freezes of the raw data
# cwd_Figures=os.path.join(cwd,'Figures')#figures and code for generating them can go here
cwd_Output=os.path.join(cwd,'Output Dataframes')

In [5]:
def GetAwardAmount(input_String, Lists):
    #function takes three arguments; the Reporter output, the destination where we store results, and an additional list for storing a freeze of the data
    GetResults = input_String.find("\"results\"") #find the part of the output detailing grant award amount, found after the "results" block of the ouput
    ResultsList = input_String[GetResults:].replace("},", "")# each grant's information is separated by curly brackets; splitting along curly brackets divides info from each grant
    ResultsList = (ResultsList.split("{\""))[1:]    #saving the individual grant amount as a an element in a list of grants
    for iGrant in ResultsList:# for each grant returned by the query
        Award_Start = iGrant.find("\"award_amount\":")#find the part detailing award amount
        Award_End = iGrant.find("\"project_start_date\":")#find the part that comes after the award amount
        DirectCost = iGrant.find("\"direct_cost_amt\":")
        direct_End = iGrant.find("\"indirect_cost_amt\":")
        Award_string = iGrant[Award_Start:Award_End].replace(",", "").split(":")[1]# the amount of money for grant will be between the part addressed as award amount and the direct cost amount
        directCost = iGrant[DirectCost:direct_End].replace(",", "").split(':', 1)[1]
        indirectCost = iGrant[direct_End:].replace(",", "").split(':', 1)[1]
        if not Award_string == "null": # for some reason, some grants do not have an award amount stored in NIH Reporter
            Lists[0] = Lists[0] + int(Award_string)
            if not directCost == "null":
                Lists[1]=Lists[1]+int(directCost)
            if not "null" in indirectCost:
                indirectCost=indirectCost.replace("}]}","")
                Lists[2]=Lists[2]+int(indirectCost)
    return Lists

In [3]:

xlsx_path = Path("ShortMeeting history.xlsx")  # <-- change if needed

# Read sheets (your first sheet name is a bit odd, so grab by index)
xls = pd.ExcelFile(xlsx_path)
participants_df = pd.read_excel(xlsx_path, sheet_name=xls.sheet_names[0])
meetings_df      = pd.read_excel(xlsx_path, sheet_name="Meetings")

participants_df.iloc[ 1:2] = "Targeting Lipid Biology in Cancer"
participants_df.tail(5)

,Participant,Meeting,Type,First Name,Last Name,Suffix,Institution,Title
54,"Elsa Flores, PhD",2005 Scholar Retreat,Scholar,Elsa,Flores,PhD,MD Anderson Cancer Center,NaN
55,"Kimryn Rathmell, MD, PhD",2005 Scholar Retreat,Scholar,Kimryn,Rathmell,"MD, PhD",Vanderbilt University Medical Center,NaN
56,"Masashi Narita, MD, PhD",2005 Scholar Retreat,Scholar,Masashi,Narita,"MD, PhD",Cambridge Institute,NaN
57,"Jan Karlseder, PhD",2005 Scholar Retreat,Scholar,Jan,Karlseder,PhD,Salk Institute,NaN
58,"James Amatruda, MD, PhD",2005 Scholar Retreat,Scholar,James,Amatruda,"MD, PhD",Memorial Sloan Kettering Cancer Center,NaN


In [4]:
# Cell 2 — helpers (normalize meeting titles + parse chair names)

def normalize_meeting_title(x: str) -> str:
    """
    Make meeting titles comparable across sheets:
    - cast to str
    - strip leading/trailing whitespace
    - remove surrounding quotes
    - collapse internal whitespace (including newlines)
    """
    if pd.isna(x):
        return None
    s = str(x).strip()
    # remove one pair of surrounding quotes if present
    if (len(s) >= 2) and ((s[0] == s[-1]) and s[0] in {"'", '"'}):
        s = s[1:-1].strip()
    s = re.sub(r"\s+", " ", s)  # collapse newlines/tabs/multiple spaces
    return s

def split_chair_names(chairs_cell) -> list[str]:
    """
    Turn the 'Meeting Chairs' cell into a list of chair name strings.
    Handles separators like ';', ',', ' and ', '&', and common ' of ' patterns.
    Keeps credentials as part of the name string (e.g., 'MD, PhD').
    """
    if pd.isna(chairs_cell):
        return []
    s = str(chairs_cell).strip()
    s = re.sub(r"\s+", " ", s)

    # Many entries look like "Name of Institution; Name of Institution"
    # Split primarily on ';' first.
    parts = [p.strip() for p in s.split(";") if p.strip()]

    # Further split each part on " and " / " & " if it contains multiple chairs.
    chairs = []
    for p in parts:
        sub = re.split(r"\s+(?:and|&)\s+", p)
        for item in sub:
            item = item.strip()
            if not item:
                continue
            # Remove trailing institution phrase like " of XYZ" (optional, but helps matching)
            item = re.sub(r"\s+of\s+.+$", "", item).strip()
            chairs.append(item)

    # de-dup while preserving order
    seen = set()
    out = []
    for c in chairs:
        if c not in seen:
            seen.add(c)
            out.append(c)
    return out

In [5]:
meetings_df = meetings_df.copy()
participants_df = participants_df.copy()

meetings_df["MeetingTopic_norm"] = meetings_df["Meeting Topic"].map(normalize_meeting_title)
participants_df["Meeting_norm"]  = participants_df["Meeting"].map(normalize_meeting_title)

# Map normalized meeting topic -> year (if duplicates exist, keep the first non-null year)
meeting_to_year = (
    meetings_df.dropna(subset=["MeetingTopic_norm", "Year"])
               .drop_duplicates(subset=["MeetingTopic_norm"])
               .set_index("MeetingTopic_norm")["Year"]
               .to_dict()
)

# Attach year onto participants using normalized title
participants_df["Year"] = participants_df["Meeting_norm"].map(meeting_to_year)

participants_df[["Participant", "Meeting", "Year"]]

,Participant,Meeting,Year
0,"Alison Ringel, PhD",Targeting Lipid Biology in Cancer,2023
1,Targeting Lipid Biology in Cancer,Targeting Lipid Biology in Cancer,2023
2,"Bart Vanhaesebroeck, PhD",Targeting Lipid Biology in Cancer,2023
3,"Christina Mitchell, MB BS, PhD",Targeting Lipid Biology in Cancer,2023
4,"Neil Vasan, MD, PhD",Targeting Lipid Biology in Cancer,2023
5,"Prof Banafshe Larijani , PhD",Targeting Lipid Biology in Cancer,2023
6,"Ray Blind,",Targeting Lipid Biology in Cancer,2023
7,"Brooke Emerling, PhD",Targeting Lipid Biology in Cancer,2023
8,"Gretchen Alicea, PhD",Targeting Lipid Biology in Cancer,2023
9,"Sarah Skuli,",Targeting Lipid Biology in Cancer,2023


In [6]:
# Cell 4 — (1) create dict keyed by "Meeting Topic (Year)" with list of participant full names

# Keep only rows that have a meeting + participant name
p = participants_df.dropna(subset=["Meeting_norm", "Participant"]).copy()

# Build a key string like "Targeting Lipid Biology in Cancer (2021)"
def make_meeting_year_key(meeting_norm, year):
    y = "" if pd.isna(year) else str(int(year)) if float(year).is_integer() else str(year)
    return f"{meeting_norm} ({y})" if y else f"{meeting_norm} (Year Unknown)"

p["MeetingYearKey"] = [make_meeting_year_key(m, y) for m, y in zip(p["Meeting_norm"], p["Year"])]

meeting_attendees_dict = (
    p.groupby("MeetingYearKey")["Participant"]
     .apply(lambda s: sorted(set(s.dropna().astype(str).str.strip())))
     .to_dict()
)

# Example: show first 5 keys
list(meeting_attendees_dict["Targeting Lipid Biology in Cancer (2023)"])

['Alison Ringel, PhD',
 'Bart Vanhaesebroeck, PhD',
 'Brooke Emerling, PhD',
 'Christina Mitchell, MB BS, PhD',
 'David Fruman, PhD',
 'Emilio Hirsch, PhD',
 'Gretchen Alicea, PhD',
 'Hua Eleanor Yu, PhD',
 'Jeremy Baskin, PhD',
 'Karen Dixon,',
 'Livia  Schiavinato Eberlin, PhD',
 'Neil Vasan, MD, PhD',
 'Prof Banafshe  Larijani , PhD',
 'Ray Blind,',
 'Sarah  Skuli,',
 'Tamas Balla, MD, PhD',
 'Targeting Lipid Biology in Cancer',
 'Vytas Bankaitis, PhD']

In [ ]:
RREPORTER_SEARCH_URL = "https://api.reporter.nih.gov/v2/projects/search"
_SUFFIXES = {"jr","sr","ii","iii","iv","md","phd","mph","ms","m.d.","ph.d.","dr"}

# ---------- name normalization + matching ----------

def _clean(s):
    s = re.sub(r"\([^)]*\)", "", str(s or "")).strip()
    s = re.sub(r"\s+", " ", s)
    s = re.sub(r"^(dr\.?|prof\.?)\s+", "", s, flags=re.I)
    return s

def parse_first_last(name):
    s = _clean(name)
    if not s: return "", ""
    parts = [p.strip() for p in s.split(",") if p.strip()]
    if len(parts) >= 2:  # "Last, First ..."
        last = parts[0]
        first = parts[1].split()[0] if parts[1] else ""
        return first.lower(), last.lower()
    toks = [t for t in s.split() if t.lower().strip(".") not in _SUFFIXES]
    if len(toks) == 1: return "", toks[0].lower()
    return toks[0].lower(), toks[-1].lower()

def attendee_key(name):
    first, last = parse_first_last(name)
    # key = (last, first initial) — robust to middle initials, degrees, etc.
    fi = first[0] if first else ""
    return (last, fi)

def project_pi_keys(pi_list):
    """Turn API principal_investigators list into {(last, first_initial), ...}."""
    keys = set()
    if not isinstance(pi_list, list): return keys
    for pi in pi_list:
        # RePORTER usually gives objects with first_name / last_name, but handle strings too
        if isinstance(pi, dict):
            fn = str(pi.get("first_name") or pi.get("firstName") or "").strip()
            ln = str(pi.get("last_name")  or pi.get("lastName")  or "").strip()
            if not ln:
                # sometimes there's 'pi_name' or similar
                maybe = pi.get("pi_name") or pi.get("name")
                fn2, ln2 = parse_first_last(maybe)
                fn, ln = fn2, ln2
            fn, ln = fn.lower(), ln.lower()
        else:
            fn, ln = parse_first_last(pi)
        if ln:
            keys.add((ln, fn[0] if fn else ""))
    return keys

def meeting_year_from_key(meeting_key):
    m = re.search(r"\((\d{4})\)\s*$", str(meeting_key))
    return int(m.group(1)) if m else None

def five_year_bins(center_year, n_before=1, n_after=3):
    return ([(center_year-5*i, center_year-5*(i-1)) for i in range(n_before, 0, -1)] +
            [(center_year+5*i, center_year+5*(i+1)) for i in range(0, n_after)])

def fiscal_years_for_bin(start, end):
    return list(range(int(start), int(end) + 1))

# ---------- RePORTER paging ----------

def reporter_search_all_pages(payload, sleep=0.25, limit=500, verbose=False):
    params = deepcopy(payload); params.update({"offset": 0, "limit": limit})
    pages, total = [], None
    while True:
        r = requests.post(REPORTER_SEARCH_URL, json=params, timeout=60)
        r.raise_for_status()
        page = r.json(); pages.append(page)
        print(params)
        print("page")
        print(page)
        total = total or page.get("meta", {}).get("total", 0)
        off = page.get("meta", {}).get("offset", params["offset"])
        cnt = page.get("meta", {}).get("count", len(page.get("results", [])))
        if verbose:
            print(f"      page offset={off} count={cnt} total={total}")
        if cnt == 0 or off + cnt >= total: break
        params["offset"] = off + cnt
        time.sleep(sleep)
    return {"total": int(total or 0), "pages": pages}

def pick_cost(proj):
    c = proj.get("fy_total_cost")
    if c is None: c = proj.get("award_amount")
    return int(c or 0)

# ---------- main analysis ----------

def multi_pi_awards_with_2plus_attendees(
    meeting_attendees_dict,
    NIH_param_template,
    n_bins_before=1,
    n_bins_after=3,
    sleep=0.25,
    max_attendees_per_meeting=None,     # optional cap for speed
    verbose_meetings=True,
    verbose_attendees=False,
    verbose_paging=False,
):
    summary_rows, detail_rows = [], []
    print("\n=== START: multi-PI awards that include >=2 meeting attendees ===")
    print("Strategy: query each attendee individually -> inspect PI roster -> keep awards with >=2 attendee matches.\n")

    for meeting_key, attendee_names in meeting_attendees_dict.items():
        if verbose_meetings:
            print("\n===================================================")
            print(f"Meeting: {meeting_key}")
            print(f"Attendees listed: {len(attendee_names)}")

        year = meeting_year_from_key(meeting_key)
        if year is None:
            print("  ⚠ No (YYYY) found at end of meeting key; skipping.")
            continue

        # Build attendee key set for matching against awards' PI rosters
        keys = []
        for nm in attendee_names:
            k = attendee_key(nm)
            if k[0]: keys.append((nm, k))
        # de-dupe
        seen = set(); clean = []
        for nm, k in keys:
            if k in seen: continue
            seen.add(k); clean.append((nm, k))

        if max_attendees_per_meeting and len(clean) > max_attendees_per_meeting:
            clean = clean[:max_attendees_per_meeting]
            print(f"  (Capped attendees to {max_attendees_per_meeting} for runtime)")

        attendee_key_set = set(k for _, k in clean)
        print(f"Meeting year: {year} | Parsable attendee keys: {len(attendee_key_set)}")

        for start, end in five_year_bins(year, n_bins_before, n_bins_after):
            bin_label = f"{start}-{end}"
            fys = fiscal_years_for_bin(start, end)
            print(f"\n--- Bin {bin_label} (FY {fys[0]}..{fys[-1]}) ---")

            # Per-bin dedupe and tracking
            seen_appl_ids = set()              # awards counted once per bin
            kept_appl_ids = set()              # awards that have >=2 attendee matches
            total_queries = 0
            total_records_seen = 0

            # Keep best/representative record per appl_id for summary dollars
            appl_to_cost = {}                  # appl_id -> max cost seen (safer than "first")
            appl_to_pi_matches = {}            # appl_id -> list of matching attendee keys (for reporting)

            for attendee_name, attendee_k in clean:
                total_queries += 1
                if verbose_attendees:
                    print(f"  Querying attendee: {attendee_name}  key={attendee_k}")

                payload = deepcopy(NIH_param_template)
                payload.setdefault("criteria", {})
                c = payload["criteria"]
                # Query by attendee name (single) + multi-PI-only + FY bin
                # NOTE: this can return awards where this attendee is one of the PIs; we then validate co-attendee matches.
                c["pi_names"] = [{"any_name": attendee_k[0], "first_name": ""}]  # last name only; reduces misses
                c["multi_pi_only"] = True
                c["fiscal_years"] = fys
                c.pop("advanced_text_search", None)

                payload["include_fields"] = [
                    "appl_id","subproject_id","fiscal_year","project_num","project_title",
                    "fy_total_cost","award_amount","principal_investigators"
                ]

                res = reporter_search_all_pages(payload, sleep=sleep, verbose=verbose_paging)
                if res["total"] == 0:
                    continue

                # Scan results; validate that >=2 meeting attendees are on the PI roster
                for page in res["pages"]:
                    for proj in page.get("results", []):
                        total_records_seen += 1
                        appl = proj.get("appl_id")
                        if appl is None: 
                            continue

                        pi_list = proj.get("principal_investigators")
                        pi_keys = project_pi_keys(pi_list)
                        matches = sorted(attendee_key_set.intersection(pi_keys))

                        # Record per-appl info even if not kept (useful for debugging)
                        cost = pick_cost(proj)
                        if appl not in appl_to_cost:
                            appl_to_cost[appl] = cost
                        else:
                            appl_to_cost[appl] = max(appl_to_cost[appl], cost)  # avoid undercounting

                        if appl not in appl_to_pi_matches:
                            appl_to_pi_matches[appl] = matches
                        else:
                            # union the matches we’ve observed across pages/queries
                            appl_to_pi_matches[appl] = sorted(set(appl_to_pi_matches[appl]).union(matches))

                        # Keep only awards with >=2 attendee PI matches
                        if len(matches) >= 2:
                            kept_appl_ids.add(appl)

                            # Details row (we’ll still dedupe summary at appl_id)
                            detail_rows.append({
                                "MeetingYearKey": meeting_key,
                                "Bin": bin_label,
                                "ApplId": appl,
                                "FiscalYear": proj.get("fiscal_year"),
                                "ProjectNum": proj.get("project_num"),
                                "ProjectTitle": proj.get("project_title"),
                                "AwardAmount": cost,
                                "MatchedAttendeeKeys": matches,
                                "PrincipalInvestigators_raw": pi_list,
                            })

                time.sleep(sleep)

            # Per-bin counting: only count appl_ids that passed the >=2 rule, and only once
            for appl in kept_appl_ids:
                if appl in seen_appl_ids:
                    continue
                seen_appl_ids.add(appl)

            unique_awards = len(seen_appl_ids)
            total_award_amount = sum(appl_to_cost.get(appl, 0) for appl in seen_appl_ids)

            print(f"Bin results: attendee-queries={total_queries}, records_seen={total_records_seen}")
            print(f"  Awards with >=2 attendee PIs (unique appl_id): {unique_awards}")
            print(f"  Total award amount (max per appl_id): ${total_award_amount:,}")

            summary_rows.append({
                "MeetingYearKey": meeting_key,
                "MeetingYear": year,
                "Bin": bin_label,
                "Attendees_parsed": len(attendee_key_set),
                "Attendee_queries": total_queries,
                "Records_seen": total_records_seen,
                "Unique_awards_2plus_attendees": unique_awards,
                "Total_award_amount": total_award_amount
            })

    print("\n=== COMPLETE ===")
    return pd.DataFrame(summary_rows), pd.DataFrame(detail_rows)

In [25]:
NIH_param = {"criteria": {}}  # minimal template is fine

summary_df, details_df = multi_pi_awards_with_2plus_attendees(
    meeting_attendees_dict,
    NIH_param_template=NIH_param,
    n_bins_before=1,
    n_bins_after=3,
    sleep=0.25,
    max_attendees_per_meeting=None,   # set e.g. 50 if meetings are large
    verbose_meetings=True,
    verbose_attendees=False,
    verbose_paging=False
)

summary_df.sort_values(["MeetingYearKey","Bin"]).head(20)


=== START: multi-PI awards that include >=2 meeting attendees ===
Strategy: query each attendee individually -> inspect PI roster -> keep awards with >=2 attendee matches.


Meeting: 2005 Scholar Retreat (2005)
Attendees listed: 17
Meeting year: 2005 | Parsable attendee keys: 17

--- Bin 2000-2005 (FY 2000..2005) ---
page
{'meta': {'search_id': 'ykpOPROOpkO9Y_VJfjPzcA', 'total': 0, 'offset': 0, 'limit': 500, 'sort_field': None, 'sort_order': 'ASC', 'sorted_by_relevance': True, 'properties': {'URL': 'https:/reporter.nih.gov/search/ykpOPROOpkO9Y_VJfjPzcA/projects'}}, 'results': []}
page
{'meta': {'search_id': '4axAc2v4OkevR_Ol1fDcVg', 'total': 0, 'offset': 0, 'limit': 500, 'sort_field': None, 'sort_order': 'ASC', 'sorted_by_relevance': True, 'properties': {'URL': 'https:/reporter.nih.gov/search/4axAc2v4OkevR_Ol1fDcVg/projects'}}, 'results': []}
page
{'meta': {'search_id': '_WbY2v6SfE66sOCu_zb8uQ', 'total': 0, 'offset': 0, 'limit': 500, 'sort_field': None, 'sort_order': 'ASC', 'sorted_by

KeyboardInterrupt: 